# Análisis y predicción del consumo por hora

El objetivo es estudiar el comportamiento histórico del consumo horario y evaluar dos escenarios distintos:

1. **Forecast rolling de una hora:** en cada instante ya conocemos el consumo de la hora anterior.
2. **Extensión de 90 días:** todas las horas se predicen desde el último dato conocido, reutilizando después las propias predicciones.

La frecuencia de ambos resultados es horaria; lo que cambia es el horizonte disponible.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('No se encuentra la raíz del proyecto ni la carpeta src')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_energy_data
from src.evaluation import naive_from_lag, regression_metrics, split_at
from src.features import build_causal_features
from src.modeling import build_catboost, build_xgboost
from src.recursive import recursive_forecast, recursive_naive

DATA_PATH = PROJECT_ROOT / 'data' / 'energy_train.csv'
TEST_START = '2016-01-01 00:00:00'
THREE_MONTH_HOURS = 24 * 90

## 1. Calidad y preparación de la serie

La carga ordena las observaciones, conserva una fila por marca temporal, completa la rejilla horaria e interpola únicamente los huecos internos.

In [ ]:
energy, quality = load_energy_data(DATA_PATH)
print(quality)
energy.describe()

## 2. Comportamiento temporal

Primero se observa la evolución general y después se resumen los ciclos intradía, semanales y mensuales. Estos perfiles describen regularidades históricas; no son todavía predicciones.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
energy['Energy'].resample('D').mean().plot(ax=axes[0], linewidth=0.8)
energy['Energy'].resample('D').mean().rolling(365).mean().plot(ax=axes[0], linewidth=2, label='Media móvil 365 días')
axes[0].set(title='Evolución del consumo medio diario', ylabel='Consumo')
axes[0].legend()
energy.loc['2015-01-01':'2015-01-14', 'Energy'].plot(ax=axes[1])
axes[1].set(title='Detalle horario — dos semanas', ylabel='Consumo')
fig.tight_layout()
plt.show()

In [ ]:
profiles = energy.assign(
    hour=energy.index.hour,
    day_of_week=energy.index.dayofweek,
    month=energy.index.month,
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
profiles.groupby('hour')['Energy'].mean().plot(ax=axes[0], marker='o', title='Perfil medio por hora')
profiles.groupby('day_of_week')['Energy'].mean().plot(ax=axes[1], marker='o', title='Perfil por día de la semana')
profiles.groupby('month')['Energy'].mean().plot(ax=axes[2], marker='o', title='Perfil medio por mes')
for axis in axes:
    axis.set_ylabel('Consumo medio')
fig.tight_layout()
plt.show()

In [ ]:
adf_statistic, p_value, used_lags, observations, *_ = adfuller(
    energy['Energy'], maxlag=24, autolag='AIC', result_object=False
)
pd.Series({
    'ADF statistic': adf_statistic,
    'p-value': p_value,
    'lags': used_lags,
    'observations': observations,
})

Un p-valor pequeño permite rechazar la hipótesis de raíz unitaria bajo esta especificación. Eso no implica que hayan desaparecido la estacionalidad, los cambios de nivel o la dependencia temporal observados en los gráficos.

## 3. Variables causales y división temporal

Los retardos usan observaciones anteriores. Las medias y desviaciones móviles se calculan sobre `Energy.shift(1)`, de modo que el consumo de la hora objetivo nunca participa en sus propias variables.

In [ ]:
features = build_causal_features(energy)
train, test = split_at(features, TEST_START)
feature_columns = [column for column in features if column != 'target']
print(f'Train: {train.index.min()} — {train.index.max()} ({len(train):,} horas)')
print(f'Test:  {test.index.min()} — {test.index.max()} ({len(test):,} horas)')
features.head()

## 4. Forecast rolling de una hora

Cada predicción puede utilizar el consumo real de la hora anterior. Se comparan persistencia, referencia semanal, XGBoost y CatBoost. WAPE expresa el error absoluto total como porcentaje del consumo observado.

In [ ]:
x_train, y_train = train[feature_columns], train['target']
x_test, y_test = test[feature_columns], test['target']
predictions = {
    'Persistence (1h)': naive_from_lag(test, 1),
    'Seasonal naive (168h)': naive_from_lag(test, 168),
}
models = {'XGBoost': build_xgboost(), 'CatBoost': build_catboost()}
for name, model in models.items():
    model.fit(x_train, y_train)
    predictions[name] = model.predict(x_test)
rolling_metrics = pd.DataFrame([
    {'model': name, **regression_metrics(y_test, values)}
    for name, values in predictions.items()
]).sort_values('RMSE')
rolling_metrics

In [ ]:
best_rolling = rolling_metrics.iloc[0]['model']
comparison = pd.DataFrame({'Real': y_test, best_rolling: predictions[best_rolling]}, index=test.index).tail(24 * 7)
comparison.plot(figsize=(13, 5))
plt.title('Forecast rolling — última semana de test')
plt.ylabel('Consumo')
plt.show()

## 5. Extensión: forecast horario de 90 días

Se reservan las últimas 2.160 horas. Los modelos se entrenan únicamente con el periodo anterior. Tras estimar la primera hora, los retardos y estadísticas se actualizan con las propias predicciones: ningún consumo real del horizonte futuro entra en las variables.

In [ ]:
future_index = energy.index[-THREE_MONTH_HOURS:]
recursive_train = features.loc[features.index < future_index.min()]
history = energy.loc[energy.index < future_index.min(), 'Energy']
future_actual = energy.loc[future_index, 'Energy']
recursive_predictions = {
    'Persistence': recursive_naive(history, len(future_index), lag=1),
    'Seasonal naive (168h)': recursive_naive(history, len(future_index), lag=168),
}
recursive_models = {'XGBoost': build_xgboost(), 'CatBoost': build_catboost()}
for name, model in recursive_models.items():
    model.fit(recursive_train[feature_columns], recursive_train['target'])
    recursive_predictions[name] = recursive_forecast(model, history, future_index)
three_month_metrics = pd.DataFrame([
    {'model': name, **regression_metrics(future_actual, values)}
    for name, values in recursive_predictions.items()
]).sort_values('RMSE')
three_month_metrics

In [ ]:
best_recursive = three_month_metrics.iloc[0]['model']
pd.DataFrame({
    'Real': future_actual,
    best_recursive: recursive_predictions[best_recursive],
}, index=future_index).plot(figsize=(13, 5), linewidth=1)
plt.title('Forecast recursivo de consumo por hora — 90 días')
plt.ylabel('Consumo')
plt.show()

## 6. Conclusiones y límites

El forecast rolling mide la precisión cuando cada hora aporta una observación nueva. La extensión de 90 días es más exigente porque el error se propaga y no utiliza consumos reales intermedios. CatBoost conserva mejor el patrón horario en ese escenario, aunque suaviza picos no explicados por el calendario.

La serie no incluye temperatura, festivos ni actividad económica; estas variables serían especialmente importantes para mejorar los picos del horizonte largo. La fuente original y la unidad física tampoco están suficientemente documentadas.